In [ ]:
%%capture
!uv pip install polars

In [ ]:
import polars as pl

In [ ]:
df = pl.read_csv("../.data/chess_games_2025-01-15.csv", null_values=["None"])

In [ ]:
df.head()

In [ ]:
test_game = df.select("PGN")[0].item()

In [ ]:
tokens = test_game.encode("utf-8")
token_ids = list(map(int, tokens))

print(test_game)
print("length: ", len(test_game))
print(token_ids)
print("length:", len(token_ids))

In [ ]:
# get the most common pair of tokens
def find_token_pairs(ids):
    counts = {}

    # get pairs of tokens
    for pair in zip(ids, ids[1:]):
        if pair not in counts:
            counts[pair] = 1
        else:
            counts[pair] += 1

    return counts

In [ ]:
data_start = "<|startgame|>"
data_end = "<|endgame|>"

data_start_tokens = data_start.encode("utf-8")
data_end_tokens = data_end.encode("utf-8")

data_start_token_ids = list(map(int, data_start_tokens))
data_end_token_ids = list(map(int, data_end_tokens))

print(data_start_token_ids)
print(data_end_token_ids)

tokenizer_special_tokens = {
    "startgame": data_start,
    "endgame": data_end,
}

In [ ]:
token_pairs = find_token_pairs(token_ids)

# print(tokenPairs)
print(sorted(((v, k) for k, v in token_pairs.items()), reverse=True))

In [ ]:
top_pair = max(token_pairs, key=token_pairs.get)
top_pair

In [ ]:
def merge_pairs(ids, pair, idx):
    new_ids = []

    i = 0

    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            new_ids.append(idx)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1

        return new_ids

In [ ]:
tokens2 = merge_pairs(token_ids, top_pair, 256)

print(tokens2)
print("length:", len(tokens2))

In [ ]:
sample = df.select("PGN").sample(n=500)

In [ ]:
text = ""

for game in sample.iter_rows():
    if game[0]:
        text += game[0].strip() + "\n"

In [ ]:
text_tokens = text.encode("utf-8")
text_tokens = list(map(int, text_tokens))

In [ ]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts


def merge_pairs(ids, pair, idx):
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            new_ids.append(idx)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids


vocab_size = 512
number_of_merges = vocab_size - 256
ids = list(text_tokens)

merges = {}

for i in range(number_of_merges):
    stats = get_stats(ids)
    pair = max(stats, key=stats.get)
    idx = 256 + i
    print(f"merge {pair} into a new token {idx}")
    ids = merge_pairs(ids, pair, idx)
    merges[pair] = idx

In [ ]:
print("text length:", len(text))
print("tokens length:", len(text_tokens))
print("ids length:", len(ids))
print(f"compression ratio: {len(text_tokens) / len(ids):.2f}")

In [ ]:
vocab = {idx: bytes([idx]) for idx in range(256)}

for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]


def decode(ids: list[int]) -> str:
    tokens = b"".join(vocab[idx] for idx in ids)

    text = tokens.decode("utf-8", errors="replace")

    return text


decode(ids)

In [ ]:
def encode(text: str) -> list[int]:
    tokens = list(text.encode("utf-8"))

    while len(tokens) >= 2:
        stats = get_stats(tokens)

        pair = min(stats, key=lambda p: merges.get(p, float("inf")))

        if pair not in merges:
            break

        idx = merges[pair]

        tokens = merge_pairs(tokens, pair, idx)

    return tokens


print(encode("1.d4 d5 2.Nf3 Bf5"))

In [ ]:
print(decode(encode("1.d4 d5 2.Nf3 Bf5")))

In [ ]:
new_sample = df.select("PGN").sample(n=1)[0].item().strip()

print(new_sample == decode(encode(new_sample)))

In [ ]:
import re

# ignore `1.`, ` 2.`, ` `, etc. and get the actual moves as separate entries
tokenizer_pattern = re.compile(
    r""" ?\d+\.|\. ?| ?[-\w]+|[#+]|\s+""", flags=re.IGNORECASE
)

print(re.findall(tokenizer_pattern, new_sample))

In [ ]:
def encode_with_regex_splits(text: str) -> list[int]:
    bpe_tokens = []

    for tokens in re.findall(tokenizer_pattern, text):
        token_ids = list(tokens.encode("utf-8"))

        while len(token_ids) >= 2:
            stats = get_stats(token_ids)

            pair = min(stats, key=lambda p: merges.get(p, float("inf")))

            if pair not in merges:
                break

            idx = merges[pair]

            token_ids = merge_pairs(token_ids, pair, idx)

        bpe_tokens.extend(token_ids)

    return bpe_tokens


print(encode_with_regex_splits("1.d4 d5 2.Nf3 Bf5"))

encoded = encode_with_regex_splits("1.d4 d5 2.Nf3 Bf5")

for idx in encoded:
    print(vocab[idx].decode("utf-8"))

In [ ]:
text_chunks = re.findall(tokenizer_pattern, text)


def get_stats(ids: list, counts: dict = None) -> dict:
    counts = {} if counts is None else counts

    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1

    return counts


def merge_pairs(ids, pair, idx):
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            new_ids.append(idx)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids


vocab_size = 3072
number_of_merges = vocab_size - 256
ids = [list(ch.encode("utf-8")) for ch in text_chunks]

merges = {}
vocab = {idx: bytes([idx]) for idx in range(256)}

for i in range(number_of_merges):
    stats = {}

    for chunk_ids in ids:
        get_stats(chunk_ids, stats)

    if not stats or len(stats) < 1:
        break

    pair = max(stats, key=stats.get)

    idx = 256 + i

    ids = [merge_pairs(chunk_ids, pair, idx) for chunk_ids in ids]

    merges[pair] = idx

    vocab[idx] = vocab[pair[0]] + vocab[pair[1]]

    print(
        f"merge{i+1}/{number_of_merges} {pair} -> {idx} (\"{vocab[idx].decode('utf-8')}\": {stats[pair]} occurences)"
    )

In [ ]:
def encode_with_regex_splits(text: str) -> list[int]:
    bpe_tokens = []

    for tokens in re.findall(tokenizer_pattern, text):
        token_ids = list(tokens.encode("utf-8"))

        while len(token_ids) >= 2:
            stats = get_stats(token_ids)

            pair = min(stats, key=lambda p: merges.get(p, float("inf")))

            if pair not in merges:
                break

            idx = merges[pair]

            token_ids = merge_pairs(token_ids, pair, idx)

        bpe_tokens.extend(token_ids)

    return bpe_tokens


encoded = encode_with_regex_splits("1.d4 d5 2.Nf3 Bf5")
print(encoded)

for idx in encoded:
    print(f'"{vocab[idx].decode("utf-8")}"', end=" ")

print("\n---\n")

encoded2 = encode_with_regex_splits(new_sample)
print(encoded2)

for idx in encoded2:
    print(f'"{vocab[idx].decode("utf-8")}"', end=" ")